# C++ complexity: a complete research walkthrough

Run in a fresh environment installed with `python -m pip install -e ".[dev]"`; add Jupyter and matplotlib for this walkthrough. Never share this environment with other repositories using a generic `src` package. This notebook calls production code; source is never compiled or executed. All code cells are initially unexecuted, so no output is fabricated.

TASTY is a real, small, scraped corpus with coarse expert labels and imperfect family metadata. Classifier probabilities are uncalibrated estimates, not Big-O proofs. Data/weights remain ignored. Historical test outcomes were observed; notebook defaults never rescore them.

In [ ]:
from pathlib import Path
import os, json, subprocess, sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Open this notebook from its own repository'
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
if str(ROOT / 'src') not in sys.path: sys.path.insert(0, str(ROOT / 'src'))
RUN_FETCH = False
RUN_TRAIN = False
RUN_INFERENCE = False
print('Repository:', ROOT)
print('Python:', sys.version.split()[0])


## Acquire and inspect real inputs
The production adapter verifies the pinned source revision/SHA and strips comments before modeling. Enable downloads only into a clean checkout without historical TASTY artifacts. Read the license/provenance caveats before reuse.

In [ ]:
DATA = Path(os.environ.get('COMPLEXITY_DATA', 'data/processed/tasty-time_complexity.jsonl'))
if RUN_FETCH:
    assert not DATA.exists() and not Path('data/external/tasty.csv').exists(), 'Preserve cached historical source files'
    subprocess.run([sys.executable, '-m', 'src.data.tasty', '--output', str(DATA)], check=True)
if DATA.exists():
    from collections import Counter
    records = [json.loads(line) for line in DATA.read_text(encoding='utf-8').splitlines() if line.strip()]
    print('Real rows:', len(records))
    print('Labels:', dict(Counter(row['label'] for row in records)))
else:
    print('Enable acquisition or set COMPLEXITY_DATA to an ignored cached dataset.')


## Train on grouped development data only
The family-weight study uses fixed problem/clone grouping and a prespecified dual validation gate. It includes the retained v2 comparator. The production study expects the documented cached dataset and split paths in this repository; changing them requires a separately registered protocol. A unique output directory preserves historical reports. Training is optional and never scores the observed holdout.

In [ ]:
from datetime import datetime, timezone
if RUN_TRAIN:
    output = Path('models') / ('notebook-family-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
    assert not output.exists()
    subprocess.run([sys.executable, 'train.py', 'family', '--output', str(output)], check=True)


## Compare recorded validation evidence
These are committed aggregate results, not a fresh notebook execution. The gate rejected all added mechanisms/family-weight candidates; the v2 model remains default. The holdout is intentionally excluded from this comparison.

In [ ]:
report = json.loads(Path('models/family-weight-v4/results.json').read_text(encoding='utf-8'))
summary = [(row['candidate'], row['oof']['macro_f1'], row['validation']['macro_f1']) for row in report['runs']]
print('candidate, grouped OOF macro-F1, validation macro-F1')
for row in summary: print(row)


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar([row[0] for row in summary], [row[1] for row in summary])
ax.set(ylabel='Grouped OOF macro-F1', title='Recorded development comparison')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout()


## Saved-model inference
Weights are deliberately not distributed. Set `COMPLEXITY_MODEL` to your local retained checkpoint directory or a compatible newly trained release. This uses production inference on the shipped example and does not access any corpus label.

In [ ]:
if RUN_INFERENCE:
    from src.models.predict_model import predict
    model = Path(os.environ.get('COMPLEXITY_MODEL', 'models/structure-v2-weights'))
    assert (model / 'model.joblib').exists(), 'Train or provide ignored weights first'
    print(json.dumps(predict(Path('examples/linear.cpp').read_text(encoding='utf-8'), model), indent=2))


## Interpretation
Rare classes, imperfect family IDs and source overlap limit inference. A high validation score alone did not override the grouped OOF adoption gate. See `Report.md` and the versioned plans for all rejected candidates. An independent larger expert-labeled C++ source is needed for stronger generalization evidence.